# ShopEase Demo 1: Prompting Strategies with Groq

Compare zero-shot, few-shot, and step-guided (chain-of-thought style) prompting on the same refund-policy question. We use Groq's OpenAI-compatible chat completions API directly; OpenRouter and Claude are not used.

**Run the notebook:** select the Python 3 kernel and run all cells. With `GROQ_API_KEY` set before starting Jupyter, cells call Groq. Without a key, the notebook runs an explicitly labeled offline preview so you can teach the flow without a live API call.

To use live Groq, set `$env:GROQ_API_KEY="your-key"` in PowerShell before launching VS Code/Jupyter. Keep the key out of the notebook.

In [1]:
import os
from pathlib import Path
from groq import Groq

def find_groq_key():
    if os.getenv("GROQ_API_KEY"):
        return os.environ["GROQ_API_KEY"]
    # Optionally read a local Module_3/.env file; the notebook never stores the key.
    for folder in (Path.cwd(), *Path.cwd().parents):
        env_file = folder / ".env"
        if env_file.is_file():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line.startswith("GROQ_API_KEY="):
                    value = line.split("=", 1)[1].strip().strip("\"'")
                    if value and value != "your_groq_key_here":
                        return value
    return None

MODEL = "openai/gpt-oss-120b"
GROQ_API_KEY = find_groq_key()
LIVE_API = bool(GROQ_API_KEY)
client = Groq(api_key=GROQ_API_KEY) if LIVE_API else None
print("Mode:", "LIVE Groq API" if LIVE_API else "OFFLINE PREVIEW (no API request)")

Mode: LIVE Groq API


## Policy and one customer question

The experiment changes only the prompt style. We compare each answer with the policy outcome. The token counts shown are the model provider's usage data when live; offline preview values are illustrative.

In [2]:
RETURN_POLICY = 'ShopEase Return & Refund Policy:\n1. Items can be returned within 30 days of delivery for a full refund.\n2. Days 31-45: store credit only, and only if unopened/unused in original packaging.\n3. After day 45, no returns, except items damaged/defective on arrival reported within 90 days.\n4. Opened electronics past day 30 are not eligible for refund or credit.\n5. Final sale / clearance items are never eligible.'
QUESTION = "I bought a blender 35 days ago. The box was opened and it stopped working after two uses. Can I get a refund?"
EXPECTED = "Not eligible: opened electronics past day 30 are excluded by the policy."
print(QUESTION)
print("Expected policy outcome:", EXPECTED)

I bought a blender 35 days ago. The box was opened and it stopped working after two uses. Can I get a refund?
Expected policy outcome: Not eligible: opened electronics past day 30 are excluded by the policy.


In [3]:
PROMPTS = {
    "Zero-shot": lambda q: f"Customer question: {q}\n\nIs the customer eligible for refund, store credit, or neither? Answer clearly.",
    "Few-shot": lambda q: (
        "Examples:\n"
        "Q: Unopened shoes returned 10 days after delivery.\nA: Eligible for a full refund: within 30 days and unopened.\n"
        "Q: Clearance jacket returned after 5 days.\nA: Not eligible: clearance items are final sale.\n\n"
        f"Now answer this question using the same policy: {q}"
    ),
    "Step-guided": lambda q: (
        f"Customer question: {q}\n\n"
        "Apply the policy in order: identify days since delivery, whether the item is opened, "
        "whether it is electronics, then state the applicable rule and final verdict.\n\n"
        f"Policy:\n{RETURN_POLICY}"
    ),
}

def ask(prompt):
    if not LIVE_API:
        return {"answer": EXPECTED + " (offline example)", "tokens": "illustrative"}
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a ShopEase policy assistant. Follow only the supplied return policy.\n\n" + RETURN_POLICY},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
        max_tokens=300,
    )
    return {"answer": response.choices[0].message.content, "tokens": response.usage.total_tokens}

results = []
for strategy, build_prompt in PROMPTS.items():
    result = ask(build_prompt(QUESTION))
    results.append({"Strategy": strategy, **result})

for row in results:
    print(f"\n{row['Strategy']} ({row['tokens']} tokens)")
    print(row["answer"])


Zero-shot (533 tokens)
**Eligibility:** Neither a refund nor store credit.

**Reasoning:**  
- The purchase is 35 days old, which falls in the 31‑45‑day window where only **store credit** is offered **and only if the item is unopened and unused**.  
- The blender’s box was opened, the product was used, and it’s an electronic item. According to the policy, **opened electronics past day 30 are not eligible for any refund or credit**.  
- The defect occurred after use, not on arrival, so the “defective on arrival” exception does not apply.

Therefore, the customer does not

Few-shot (573 tokens)
**Not eligible:** The blender is an opened electronic item, and it’s now 35 days after delivery.  

- For days 31‑45 the policy allows only **store credit**, and **only if the item is unopened and unused** in its original packaging.  
- Additionally, **opened electronics past day 30 are not eligible for either a refund or store credit**.  

Since the blender was opened and used, it does not meet 

## Discussion

Compare how examples and explicit steps shape the answer. In live mode, outputs can vary between runs; evaluate the answer against policy rather than assuming one prompt style always wins.